<h1><b>Coordinate Reference Systems</b></h1>
It's pretty amazing that we can represent the Earth's surface in 2 dimensions!

# <h2 id="Introduction" tabindex="0">Introduction</h2>

<p>The maps you create in this course portray the surface of the earth in two dimensions.  But, as you know, the world is actually a three-dimensional globe. So we have to use a method called a <strong>map projection</strong> to render it as a flat surface.</p>
<p>Map projections can't be 100% accurate.  Each projection distorts the surface of the Earth in some way, while retaining some useful property.  For instance,</p>

<ul>
<li>the <em>equal-area</em> projections (like "Lambert Cylindrical Equal Area", or "Africa Albers Equal Area Conic") preserve area.  This is a good choice, if you'd like to calculate the area of a country or city, for example.</li>
<li>the <em>equidistant</em> projections (like "Azimuthal Equidistant projection") preserve distance.  This would be a good choice for calculating flight distance.</li>
</ul>

<center>
<img src="https://storage.googleapis.com/kaggle-media/learn/images/noBRRNR.png" width="700"><br>
<b>List of map projections</b> (<a href="https://bit.ly/2kOHTBs" rel=" noreferrer nofollow">Source</a>)<br><br>
</center>

<p>We use a <strong>coordinate reference system (CRS)</strong> to show how the projected points correspond to real locations on Earth.  In this tutorial, you'll learn more about coordinate reference systems, along with how to use them in GeoPandas.</p>

In [1]:
import geopandas as gpd
import pandas as pd

In [3]:
#!/bin/bash
!curl -L -o geospatial-learn-course-data.zip https://www.kaggle.com/api/v1/datasets/download/alexisbcook/geospatial-learn-course-data
!ls
!unzip geospatial-learn-course-data.zip -d geospatial-learn-course-data

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  233M  100  233M    0     0  25.0M      0  0:00:09  0:00:09 --:--:-- 30.0M
geospatial-learn-course-data.zip  sample_data
Archive:  geospatial-learn-course-data.zip
  inflating: geospatial-learn-course-data/CA_county_boundaries/CA_county_boundaries/CA_county_boundaries.cpg  
  inflating: geospatial-learn-course-data/CA_county_boundaries/CA_county_boundaries/CA_county_boundaries.dbf  
  inflating: geospatial-learn-course-data/CA_county_boundaries/CA_county_boundaries/CA_county_boundaries.prj  
  inflating: geospatial-learn-course-data/CA_county_boundaries/CA_county_boundaries/CA_county_boundaries.shp  
  inflating: geospatial-learn-course-data/CA_county_boundaries/CA_county_boundaries/CA_county_boundaries.shx  
  inflating: geospatial-learn-course-dat

# <h2 id="Setting-the-CRS" tabindex="0">Setting the CRS</h2>

<p>When we create a GeoDataFrame from a shapefile, the CRS is already imported for us.</p>

In [4]:
# Load a GeoDataFrame containing regions in Ghana
regions = gpd.read_file("geospatial-learn-course-data/ghana/ghana/Regions/Map_of_Regions_in_Ghana.shp")
print(regions.crs)

EPSG:32630


<p>How do you interpret that?</p>

<p>Coordinate reference systems are referenced by <a href="http://www.epsg.org/">European Petroleum Survey Group (EPSG)</a> codes.</p>

<p>This GeoDataFrame uses <a href="https://epsg.io/32630">EPSG 32630</a>, which is more commonly called the "Mercator" projection. This projection preserves angles (making it useful for sea navigation) and slightly distorts area.</p>

<p>However, when creating a GeoDataFrame from a CSV file, we have to set the CRS.  <a href="https://epsg.io/4326">EPSG 4326</a> corresponds to coordinates in latitude and longitude.</p>

In [5]:
# Create a DataFrame with health facilities in Ghana
facilities_df = pd.read_csv("geospatial-learn-course-data/ghana/ghana/health_facilities.csv")

# Convert the DataFrame to a GeoDataFrame
facilities = gpd.GeoDataFrame(facilities_df, geometry=gpd.points_from_xy(facilities_df.Longitude, facilities_df.Latitude))

# Set the coordinate reference system (CRS) to EPSG 4326
facilities.crs = {'init': 'epsg:4326'}

# View the first five rows of the GeoDataFrame
facilities.head()

/usr/local/lib/python3.12/dist-packages/pyproj/crs/crs.py:143: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When making the change, be mindful of axis order changes: https://pyproj4.github.io/pyproj/stable/gotchas.html#axis-order-changes-in-proj-6
  in_crs_string = _prepare_from_proj_string(in_crs_string)


,Region,District,FacilityName,Type,Town,Ownership,Latitude,Longitude,geometry
0,Ashanti,Offinso North,A.M.E Zion Clinic,Clinic,Afrancho,CHAG,7.40801,-1.96317,POINT (-1.96317 7.40801)
1,Ashanti,Bekwai Municipal,Abenkyiman Clinic,Clinic,Anwiankwanta,Private,6.46312,-1.58592,POINT (-1.58592 6.46312)
2,Ashanti,Adansi North,Aboabo Health Centre,Health Centre,Aboabo No 2,Government,6.22393,-1.34982,POINT (-1.34982 6.22393)
3,Ashanti,Afigya-Kwabre,Aboabogya Health Centre,Health Centre,Aboabogya,Government,6.84177,-1.61098,POINT (-1.61098 6.84177)
4,Ashanti,Kwabre,Aboaso Health Centre,Health Centre,Aboaso,Government,6.84177,-1.61098,POINT (-1.61098 6.84177)


<p>In the code cell above, to create a GeoDataFrame from a CSV file, we needed to use both Pandas and GeoPandas:</p>

<ul>
<li>We begin by creating a DataFrame containing columns with latitude and longitude coordinates.</li>
<li>To convert it to a GeoDataFrame, we use <code>gpd.GeoDataFrame()</code>.  </li>
<li>The <code>gpd.points_from_xy()</code> function creates <code>Point</code> objects from the latitude and longitude columns.</li>
</ul>

# <h2>Re-projecting</h2>

<p>Re-projecting refers to the process of changing the CRS.  This is done in GeoPandas with the <code>to_crs()</code> method.</p>
<p>When plotting multiple GeoDataFrames, it's important that they all use the same CRS.  In the code cell below, we change the CRS of the <code>facilities</code> GeoDataFrame to match the CRS of <code>regions</code> before plotting it.</p>

In [ ]:
# Create a map
ax = regions.plot(figsize=(8,8), color='whitesmoke', linestyle=':', edgecolor='black')
facilities.to_crs(epsg=32630).plot(markersize=1, ax=ax)